# 17 · 脱敏外部输出

把对外文件里的私有路径、机器名替换成占位符。

**源文件**：`src/sanitize_external_outputs.py`

逐格 `Shift+Enter`。第 1 格是导入，第 2 格是参数（路径已填好），之后是脚本主体。

In [ ]:
# ===== 导入与模块级定义 =====
#!/usr/bin/env python3
"""Remove linkable row keys from aggregate validation outputs before delivery."""

from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]

def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

def atomic_json(payload: dict, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    os.replace(temporary, path)

def replace_focal_key(path: Path) -> None:
    frame = pd.read_csv(path)
    if "focal_key" not in frame.columns:
        return
    frame.insert(0, "check_id", pd.factorize(frame["focal_key"], sort=False)[0] + 1)
    frame = frame.drop(columns=["focal_key"])
    atomic_csv(frame, path)

In [ ]:
# ===== 参数：自动定位项目根目录（换台电脑也能跑）=====
import sys
from pathlib import Path

def find_project_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / '3_分析步骤').is_dir() and (c / '5_最终交付包').is_dir():
            return c
    raise RuntimeError('找不到项目根目录。请确认这个 notebook 是在 3_分析步骤/ 文件夹里打开的。')

ROOT     = find_project_root()
DATA_DIR = ROOT / '1_题目与数据'
WORK_DIR = ROOT / '4_中间产物'
OUT_DIR  = WORK_DIR / '我算出来的结果'
print('项目根目录:', ROOT)

sys.argv = ["sanitize_external_outputs.py"]
for i, a in enumerate(sys.argv): print(f'  argv[{i}] = {a}')

In [ ]:
# ===== 第 1 段 =====
replace_focal_key(ROOT / "outputs" / "pool_identity_check.csv")
replace_focal_key(ROOT / "outputs" / "description_pool_identity_check.csv")

text_manifest_path = ROOT / "audit" / "stage2_text_manifest.json"
text_manifest = json.loads(text_manifest_path.read_text(encoding="utf-8"))
text_manifest.pop("processed_private_path", None)
text_manifest["processed_private_checkpoint"] = "omitted from external delivery"
atomic_json(text_manifest, text_manifest_path)

In [ ]:
# ===== 第 2 段 =====
engine_manifest_path = ROOT / "audit" / "stage6_engine_manifest.json"
engine_manifest = json.loads(engine_manifest_path.read_text(encoding="utf-8"))
engine_manifest["row_level_scores_persisted_in_external_zip"] = False
atomic_json(engine_manifest, engine_manifest_path)

In [ ]:
# ===== 第 3 段 =====
for filename in (
    "rq3_year_effects.csv",
    "rq4_country_effects.csv",
    "rq4_sector_effects.csv",
):
    path = ROOT / "outputs" / filename
    frame = pd.read_csv(path)
    frame["scenario_unit"] = (
        "percent change in conditional geometric mean of (1 + funding hours)"
    )
    atomic_csv(frame, path)

In [ ]:
# ===== 第 4 段 =====
print("Sanitized validation keys, metadata and scenario-unit labels for external delivery.")
#     return 0